In [7]:
import fitz
import faiss
import numpy as np
import os

from sentence_transformers import SentenceTransformer
from openai import OpenAI
from dotenv import load_dotenv

In [ ]:
load_dotenv()

api_key = os.getenv("OPENROUTER_API_KEY")

print(api_key)

In [ ]:
client = OpenAI(
    api_key=api_key,
    base_url="https://openrouter.ai/api/v1"
)

print("Client Initialized")

In [ ]:
print("Loading Embedding Model...")

model = SentenceTransformer("BAAI/bge-small-en-v1.5")

print("Embedding Model Loaded")

In [ ]:
pdf_path = "../data/sample.pdf"

doc = fitz.open(pdf_path)

full_text = ""

for page in doc:
    full_text += page.get_text()

print("Characters:", len(full_text))

In [ ]:
chunk_size = 500

chunks = [
    full_text[i:i+chunk_size]
    for i in range(0, len(full_text), chunk_size)
]

print("Total Chunks:", len(chunks))

In [ ]:
embeddings = model.encode(chunks)

embeddings = embeddings.astype("float32")

faiss.normalize_L2(embeddings)

print(embeddings.shape)

In [ ]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(embeddings)

print("Vectors Stored:", index.ntotal)

In [15]:
def retrieve(query, k=5):

    query_embedding = model.encode([query]).astype("float32")

    faiss.normalize_L2(query_embedding)

    scores, indices = index.search(query_embedding, k)

    retrieved_chunks = []

    for idx in indices[0]:
        retrieved_chunks.append(chunks[idx])

    return retrieved_chunks

In [ ]:
results = retrieve("What is SpectralHashNetV6?")

print(results[0])

In [19]:
def ask_rag(question):

    retrieved_docs = retrieve(question)

    context = "\n\n".join(retrieved_docs)

    prompt = f"""
You are an AI Research Assistant.

Answer ONLY using the context provided below.

If the answer is not present, say:
"I couldn't find that information in the document."

-------------------------

Context:

{context}

-------------------------

Question:

{question}
"""

    response = client.chat.completions.create(
    model="google/gemini-2.5-flash",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],
    max_tokens=500,
    temperature=0.2
)

    return response.choices[0].message.content

In [ ]:
answer = ask_rag(
    "What is SpectralHashNetV6?"
)

print(answer)